### get OSM power lines data

In [ ]:
import yaml
import osmnx as ox
import geopandas as gpd
from pathlib import Path

# load config from project root
config_path = Path("..") / "config.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

bbox = config["area"]["bbox"]  # [min_lon, min_lat, max_lon, max_lat]
print("Area:", config["area"]["name"])
print("Bbox:", bbox)


Area: Tampere region, Finland
Bbox: [23.5, 61.3, 24.1, 61.7]


In [5]:
# fetch power lines from OSM within our bbox
# osmnx 2.x bbox order: (min_lon, min_lat, max_lon, max_lat) — same as config
min_lon, min_lat, max_lon, max_lat = bbox

lines = ox.features_from_bbox(
    bbox=(min_lon, min_lat, max_lon, max_lat),
    tags={"power": "line"}
)

print(f"Fetched {len(lines)} power line features")
print(lines.geometry.geom_type.value_counts())


Fetched 140 power line features
LineString    140
Name: count, dtype: int64


In [7]:
lines.head()

geometry power  \
element id                                                                  
way     5031651   LINESTRING (24.24837 61.35677, 24.24636 61.357...  line   
        7959145   LINESTRING (23.86311 61.45921, 23.86394 61.459...  line   
        23804410  LINESTRING (24.16276 63.03224, 24.16323 63.031...  line   
        29990678   LINESTRING (23.89332 61.5016, 23.89759 61.50412)  line   
        30167723  LINESTRING (23.88091 61.49984, 23.8827 61.4998...  line   

                 source      created_by                  name cables voltage  \
element id                                                                     
way     5031651     NaN             NaN                   NaN      3  110000   
        7959145     NaN             NaN                   NaN      6  110000   
        23804410    NaN             NaN  Alajärvi - Kangasala      3  400000   
        29990678    NaN  Potlatch 0.10f                   NaN    NaN     NaN   
        30167723    NaN             NaN                   NaN      3  110000   

                 frequency                       operator operator:wikidata  \
element id                                                                    
way     5031651        NaN                         Elenia         Q11857327   
        7959145        NaN                            NaN               NaN   
        23804410       NaN                        Fingrid          Q1417185   
        29990678       NaN                            NaN               NaN   
        30167723       NaN  Tampereen Energia Sähköverkko               NaN   

                 operator:wikipedia circuits   ref short_name   wires  \
element id                                                              
way     5031651           fi:Elenia      NaN   NaN        NaN     NaN   
        7959145                 NaN        2   NaN        NaN     NaN   
        23804410         fi:Fingrid      NaN  1097      AJ-KA  double   
        29990678                NaN      NaN   NaN        NaN     NaN   
        30167723                NaN      NaN   NaN        NaN  single   

                   start_date line  
element id                          
way     5031651           NaN  NaN  
        7959145           NaN  NaN  
        23804410          NaN  NaN  
        29990678          NaN  NaN  
        30167723  before 1974  NaN

In [8]:
from shapely.geometry import box

# keep only the columns useful for analysis
cols_to_keep = ["geometry", "voltage", "cables", "circuits", "wires", "operator", "name", "ref"]
lines_clean = lines[[c for c in cols_to_keep if c in lines.columns]].copy()

# clip to our exact bbox — removes features that only partially overlapped
bbox_polygon = box(min_lon, min_lat, max_lon, max_lat)
lines_clean = lines_clean.clip(bbox_polygon)

# save to GeoPackage
output_path = Path("..") / "data" / "raw" / "power_lines.gpkg"
lines_clean.to_file(output_path, driver="GPKG")

print(f"Saved {len(lines_clean)} features to {output_path}")


Saved 140 features to ../data/raw/power_lines.gpkg


In [9]:
import sys
sys.path.append("..")  # so Python can find the src/ folder

from src.corridor import create_corridor_buffer

buffer_m = config["corridor"]["buffer_m"]  # 50m from config.yaml
corridor = create_corridor_buffer(lines_clean, buffer_m)

print(f"Corridor CRS: {corridor.crs}")
print(f"Corridor geometry types: {corridor.geometry.geom_type.value_counts()}")


Corridor CRS: EPSG:4326
Corridor geometry types: Polygon    140
Name: count, dtype: int64


In [10]:
corridor_path = Path("..") / "data" / "processed" / "corridor_buffer.gpkg"
corridor.to_file(corridor_path, driver="GPKG")

print(f"Saved corridor buffer to {corridor_path}")


Saved corridor buffer to ../data/processed/corridor_buffer.gpkg


### check if sentinel2 data exist in area

In [11]:
import pystac_client
import planetary_computer

# connect to Planetary Computer STAC API
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

# search Sentinel-2 L2A scenes over our bbox — summer window for best vegetation signal
min_lon, min_lat, max_lon, max_lat = bbox

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=[min_lon, min_lat, max_lon, max_lat],
    datetime="2023-06-01/2023-08-31",
    query={"eo:cloud_cover": {"lt": config["sentinel2"]["cloud_cover_max"]}},
)

items = list(search.items())
print(f"Found {len(items)} Sentinel-2 scenes")


Found 38 Sentinel-2 scenes


In [12]:
# sort scenes by cloud cover (ascending) and show the top 5
items_sorted = sorted(items, key=lambda x: x.properties["eo:cloud_cover"])

for item in items_sorted[:5]:
    print(f"Date: {item.datetime.date()}  Cloud cover: {item.properties['eo:cloud_cover']:.1f}%  ID: {item.id}")


Date: 2023-06-19  Cloud cover: 0.0%  ID: S2A_MSIL2A_20230619T100031_R122_T34VFP_20240911T192325
Date: 2023-06-19  Cloud cover: 0.0%  ID: S2A_MSIL2A_20230619T100031_R122_T35VLJ_20240911T192325
Date: 2023-06-19  Cloud cover: 0.0%  ID: S2A_MSIL2A_20230619T100031_R122_T35VLJ_20230619T172736
Date: 2023-06-19  Cloud cover: 0.0%  ID: S2A_MSIL2A_20230619T100031_R122_T34VFP_20230619T172734
Date: 2023-06-19  Cloud cover: 0.0%  ID: S2A_MSIL2A_20230619T100031_R122_T34VFN_20230619T172735


In [13]:
# pick the best scene — lowest cloud cover, most recent reprocessing
best_item = items_sorted[0]
print(f"Selected scene: {best_item.id}")
print(f"Date: {best_item.datetime.date()}")
print(f"Cloud cover: {best_item.properties['eo:cloud_cover']}%")
print(f"Available bands: {list(best_item.assets.keys())}")


Selected scene: S2A_MSIL2A_20230619T100031_R122_T34VFP_20240911T192325
Date: 2023-06-19
Cloud cover: 0.000355%
Available bands: ['AOT', 'B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09', 'B11', 'B12', 'B8A', 'SCL', 'WVP', 'visual', 'safe-manifest', 'granule-metadata', 'inspire-metadata', 'product-metadata', 'datastrip-metadata', 'tilejson', 'rendered_preview']


In [14]:
import rioxarray

# get a signed (authenticated) URL for B04
b04_href = planetary_computer.sign(best_item.assets["B04"].href)

# open and clip to our bbox — only our area is downloaded, not the full tile
b04 = rioxarray.open_rasterio(b04_href, masked=True).rio.clip_box(
    minx=min_lon, miny=min_lat, maxx=max_lon, maxy=max_lat,
    crs="EPSG:4326"
)

print(f"Shape: {b04.shape}")       # (bands, rows, cols)
print(f"CRS: {b04.rio.crs}")
print(f"Resolution: {b04.rio.resolution()}")  # should be ~10m


Shape: (1, 4591, 3384)
CRS: EPSG:32634
Resolution: (10.0, -10.0)


In [15]:
# download the other two bands using the same pattern
b03_href = planetary_computer.sign(best_item.assets["B03"].href)
b08_href = planetary_computer.sign(best_item.assets["B08"].href)

b03 = rioxarray.open_rasterio(b03_href, masked=True).rio.clip_box(
    minx=min_lon, miny=min_lat, maxx=max_lon, maxy=max_lat, crs="EPSG:4326"
)
b08 = rioxarray.open_rasterio(b08_href, masked=True).rio.clip_box(
    minx=min_lon, miny=min_lat, maxx=max_lon, maxy=max_lat, crs="EPSG:4326"
)

# save all three bands to disk
s2_dir = Path("..") / "data" / "raw" / "sentinel2"
s2_dir.mkdir(exist_ok=True)

b04.rio.to_raster(s2_dir / "B04.tif")
b03.rio.to_raster(s2_dir / "B03.tif")
b08.rio.to_raster(s2_dir / "B08.tif")

print("Saved B03, B04, B08 to", s2_dir)


Saved B03, B04, B08 to ../data/raw/sentinel2


In [16]:
# search for Copernicus DEM tiles covering our bbox
dem_search = catalog.search(
    collections=["cop-dem-glo-30"],
    bbox=[min_lon, min_lat, max_lon, max_lat],
)

dem_items = list(dem_search.items())
print(f"Found {len(dem_items)} DEM tile(s)")
for item in dem_items:
    print(f"  {item.id}")


Found 2 DEM tile(s)
  Copernicus_DSM_COG_10_N61_00_E024_00_DEM
  Copernicus_DSM_COG_10_N61_00_E023_00_DEM


In [17]:
from rioxarray.merge import merge_arrays

# download both DEM tiles and clip to bbox
dem_tiles = []
for item in dem_items:
    href = planetary_computer.sign(item.assets["data"].href)
    tile = rioxarray.open_rasterio(href, masked=True).rio.clip_box(
        minx=min_lon, miny=min_lat, maxx=max_lon, maxy=max_lat, crs="EPSG:4326"
    )
    dem_tiles.append(tile)

# merge the two tiles into one continuous raster
dem = merge_arrays(dem_tiles)

# save to disk
dem_path = Path("..") / "data" / "raw" / "dem" / "dem_30m.tif"
dem_path.parent.mkdir(exist_ok=True)
dem.rio.to_raster(dem_path)

print(f"DEM shape: {dem.shape}")
print(f"DEM CRS: {dem.rio.crs}")
print(f"Resolution: {dem.rio.resolution()}")
print(f"Saved to {dem_path}")


DEM shape: (1, 1442, 1081)
DEM CRS: EPSG:4326
Resolution: (0.0005555555555555635, -0.00027777777777778173)
Saved to ../data/raw/dem/dem_30m.tif
